In [ ]:
!pip install -qU \
    cohere \
    langchain \
    tiktoken \
    pinecone-client \
    langchain-openai \
    langchain-pinecone \
    sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.6/166.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 973.5/973.5 kB 17.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 17.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 17.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.5/308.5 kB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.8/122.8 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 kB 39.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.9/215.9 kB 22.4 MB/s eta 0:00:00
     ━━━━━━━━━━━

In [ ]:
import os
import json
from uuid import uuid4

import numpy as np

from google.colab import userdata, drive

import tiktoken

from pinecone import Pinecone, PodSpec, ServerlessSpec, PineconeApiException

from sentence_transformers import SentenceTransformer

from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter, HTMLHeaderTextSplitter
from langchain_openai import ChatOpenAI as LangChainChatOpenAI
from langchain_openai import OpenAIEmbeddings as LangChainOpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore as LangChainPinecone



In [ ]:
def ada_embeddings(embeddings, chunks):
  embeds = embeddings.embed_documents([chunk for chunk in chunks])
  return embeds

In [ ]:
def mpnet_embeddings(model, chunks):
  embeds = model.encode([chunk for chunk in chunks])
  return embeds

In [ ]:
def wals_descriptive_embeddings():
  embeddings = LangChainOpenAIEmbeddings(model='text-embedding-ada-002',
                                         openai_api_key=userdata.get('OPENAI_KEY'))
  with open('/content/drive/MyDrive/WALS/data/descriptions_for_encoding.csv') as f:
    descriptions = [line.strip().split(',') for line in f]
  descriptive_embeddings = dict()
  for line in descriptions[:5]:
    language = line[0]
    print(language)
    descriptive_embeddings[language] = list()
    for chunk in line[1:]:
      #print(chunk)
      embeds = embeddings.embed_documents([chunk])
      #print('Embedding length: ', len(embeds[0]))
      #print(embeds[0])
      descriptive_embeddings[language] = embeds
    #print(descriptive_embeddings[language])
    descriptive_embeddings[language] = np.array(descriptive_embeddings[language])
  return descriptive_embeddings

In [ ]:
embeds = wals_descriptive_embeddings()

Albanian
Angaataha
Armenian (Eastern)
Atayal
Alladian


In [ ]:
embeds['Angaataha'].shape

(1, 1536)

In [ ]:
def load_chapter_metadata(path):
  with open(os.path.join(path)) as f:
    headers = f.readline().strip().split(',')
    lines = [line.strip().split(',') for line in f]
  chapter_metadata = dict()
  for i,line in enumerate(lines):
    chapter_metadata[i] = {'chapter number': i+1,
                          'title': line[headers.index('Title')],
                          'authors': line[headers.index('Contributors')],
                          'linguistic subfield': line[headers.index('Area')]}
  return chapter_metadata

In [ ]:
def upsert_document(index, embeddings, embed_function, filename, namespace, chapter_number=None,
                    language_name=None, split_token='\n-----\n', vector_packet_size=50):
  print(f'Reading {filename}')
  with open(f'/content/drive/MyDrive/WALS/{filename}', encoding='utf-8') as f:
    text = f.read()
  if split_token is None:
    chunks = [text]
  else:
    chunks = text.split(split_token)
  ids = [str(uuid4()) for chunk in chunks]
  embeds = embed_function(embeddings, chunks)
  #text_preview = [chunk[:1000] for chunk in chunks]
  record_metadatas = [{"chunk": j,
                       "text": chunk,
                       "chapter": chapter_number if chapter_number else 'snippet',
                       "language": language_name if language_name else 'not specified'}
                        for (j, chunk) in enumerate(chunks)]
  k = 0
  packet = list()
  for vector in zip(ids, embeds, record_metadatas):
    packet.append(vector)
    k += 1
    if k == vector_packet_size:
      try:
        index.upsert(vectors=packet, namespace=namespace)
      except PineconeApiException:
        print('Pinecone exception raised! One of the vectors is probably too large. Skipping this packet.')
      k = 0
      packet = list()

  if k > 0:
    #get any leftovers
    index.upsert(vectors=packet, namespace=namespace)


In [ ]:
def upsert_chapter_text(index, embeddings, embed_function, path, namespace='chapters', max_chapters=144):
    chapter_metadata = load_chapter_metadata('/content/drive/MyDrive/WALS/snippets/chapter_metadata.csv')
    for n in range(1, max_chapters+1):
      file = f'chapter_{n}.txt'
      with open(os.path.join(path,file), encoding='utf-8') as f:
        text = f.read()
      chunks = text.split('\n-----\n')
      print(f'On Chapter {n}')
      ids = [str(uuid4()) for _ in range(len(chunks))]
      embeds = embed_function(embeddings, chunks)
      record_metadatas = [{"chapter": n,
                             "text": chunk,}
                              #**metadata
                              #"metadata": chapter_metadata[k]}
                              for (k, chunk) in enumerate(chunks)]
      try:
        index.upsert(vectors=zip(ids, embeds, record_metadatas), namespace=namespace)
      except PineconeApiException:
        print('Pinecone exception occured. Chunk is probably bigger than 2MB! Ignoring.')

In [ ]:
def setup_vectorstore(make_new_index = False,
                      upsert_chapters = False,
                      upsert_extras = False,
                      index_name='starter-index',
                      text_embedding = 'ada'):

  pc = Pinecone(api_key=userdata.get('PINECONE_TOKEN'))

  if make_new_index == 'pod':
    #https://docs.pinecone.io/docs/manage-indexes#create-a-pod-based-index
    pc.delete_index(index_name)
    pc.create_index(name=index_name, dimension=1536, metric="cosine", spec=PodSpec(environment="gcp-starter"))
  elif make_new_index == 'serverless':
    #https://docs.pinecone.io/guides/getting-started/quickstart#4-create-a-serverless-index
    pc.delete_index(index_name)
    pc.create_index(name=index_name, dimensions=768, metric='cosine', spec=ServerlessSpec(cloud='aws', region='us-east-1'))

  index = pc.Index(index_name)

  if text_embedding == 'ada':
    embeddings = LangChainOpenAIEmbeddings(model='text-embedding-ada-002', openai_api_key=userdata.get('OPENAI_KEY'))
    embed_function = ada_embeddings
  elif text_embedding == 'mpnet':
    embeddings = SentenceTransformer('all-mpnet-base-v2')
    embed_function = mpnet_embeddings

  if upsert_chapters:
    upsert_chapter_text(index, embeddings, embed_function, '/content/drive/MyDrive/WALS/chapter_text', namespace='chapters', max_chapters=80)

  if upsert_extras:
    if upsert_extras is True:
      upsert_document(index, embeddings, embed_function, 'snippets/map_snippets.txt', namespace='snippets')
      upsert_document(index, embeddings, embed_function, 'snippets/language_snippets.txt', namespace='snippets')
      upsert_document(index, embeddings, embed_function, 'snippets/overview_details.txt', namespace='snippets')
      upsert_document(index, embeddings, embed_function, 'snippets/typology_snippets.txt', namespace='typology', split_token='*-----*')
      upsert_document(index, embeddings, embed_function, 'snippets/author_snippets.txt', namespace='authors')
    elif 'maps' in upsert_extras:
      upsert_document(index, embeddings, embed_function, 'snippets/map_snippets.txt', namespace='snippets')
    elif 'typology' in upsert_extras:
      upsert_document(index, embeddings, embed_function, 'snippets/language_snippets.txt', namespace='snippets')
      upsert_document(index, embeddings, embed_function, 'snippets/typology_snippets.txt', namespace='typology', split_token='*-----*')
    elif 'authors' in upsert_extras:
      upsert_document(index, embeddings, embed_function, 'snippets/author_snippets.txt', namespace='authors')
    elif 'summaries' in upsert_extras:
      for file in os.listdir('/content/drive/MyDrive/WALS/summaries'):
        number = file.split('_')[1]
        upsert_document(index, embeddings, embed_function, f'summaries/{file}', split_token=None, namespace='summaries', chapter_number=number)

  return index, embeddings

In [ ]:
def delete_namespace(namespace):
  pc = Pinecone(api_key=userdata.get('PINECONE_TOKEN'))
  index = pc.Index('starter-index')
  index.delete(namespace=namespace, delete_all=True)

In [ ]:
index, embeddings = setup_vectorstore(make_new_index=False,
                                      upsert_chapters=False,
                                      upsert_extras=False,
                                      index_name = 'starter-index',
                                      text_embedding= 'ada')

In [ ]:
e = ada_embeddings(embeddings, ['Tell me about possessive inflection in languages of California'])

In [ ]:
response = index.query(
    vector=e,
    top_k=5,
    include_metadata=True,
    namespace='chapters'
)

In [ ]:
for r in response['matches']:
  score = r['score']
  #if score < 0.83:
  #  continue
  print(r['metadata']['text'])
  print(f'Score:{score}')

Many languages have an opposition of two or more forms of possessive marking whose choice is conditioned not by semantics or style, but lexically, that is, on a noun-by-noun basis. This is called "posessive classification". For examples in Mesa Grande Diegueño (Yuman; California), the noun 'mother' takes the simple prefix ʔ-  'my' while 'house' takes the compound prefix ʔə-nʸ-.
Score:0.864883602
In many languages with head-marked possession (see Chapter 24) some nouns obligatorily require possessive inflection and cannot be used alone. For example, the nouns illustrated in (1) and (2) from Navajo (Athabaskan; New Mexico and Arizona) and Acoma (Keresan; New Mexico) cannot stand alone and require possessive inflection.
Score:0.863883
Languages with obligatorily possessed nouns often provide means of using these nouns independently without a possessor. Quite often the regular possessive inflectional paradigm includes an "indefinite" or "unspecified" possessor category 'someone's, somethin